# Notebook 06: Agentic Tool Use -- Function Calling with LLMs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajvermatx/careeralign.com/blob/main/genai-arch/notebooks/06-agentic-tool-use.ipynb)

## Learning Objectives

By the end of this notebook you will be able to:

1. Define tool schemas in the OpenAI function calling format
2. Execute single tool calls and feed results back to the model
3. Orchestrate multi-tool calls within a single turn
4. Handle tool results with a standardized wrapper
5. Build an observe-think-act agent loop with iteration limits
6. Implement robust error handling for tool execution (retries, logging)

---
## 1. Setup and Installation

In [ ]:
!pip install openai --quiet

---
## 2. Configuration

In [ ]:
import os
import json
import time
import random
from typing import Any, Callable
from dataclasses import dataclass, field
from functools import wraps
from openai import OpenAI

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    print("WARNING: OPENAI_API_KEY not set. Mock responses will be used.")
    print("Set it with: os.environ['OPENAI_API_KEY'] = 'your-key-here'")

client = OpenAI(api_key=api_key or "sk-mock-key")
MODEL = "gpt-4o-mini"

---
## 3. Defining Tools (Function Schemas)

Tools are defined as JSON schemas that tell the LLM what functions are available,
what parameters they accept, and what they do. The model uses these schemas to
decide when and how to call a function.

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a given city. Returns temperature, conditions, and humidity.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name, e.g. 'San Francisco'"},
                    "units": {"type": "string", "enum": ["celsius", "fahrenheit"], "description": "Temperature units"}
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_database",
            "description": "Search a product database by query string. Returns matching products with prices and ratings.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query string"},
                    "category": {"type": "string", "enum": ["electronics", "clothing", "books", "all"], "description": "Product category filter"},
                    "max_results": {"type": "integer", "description": "Maximum results to return"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression. Supports basic arithmetic and common functions (sqrt, abs, round).",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Math expression to evaluate, e.g. '2 ** 10 + 5'"}
                },
                "required": ["expression"]
            }
        }
    }
]

print(f"Defined {len(tools)} tools:")
for tool in tools:
    fn = tool["function"]
    params = list(fn["parameters"]["properties"].keys())
    print(f"  {fn['name']}({', '.join(params)})")
    print(f"    {fn['description']}")

---
## 4. Implementing Tool Functions

These are the actual Python functions that execute when the LLM requests a tool call.
In production, these would connect to real APIs and databases.

In [ ]:
import math

def get_weather(city: str, units: str = "celsius") -> dict:
    """Simulate a weather API call."""
    weather_data = {
        "San Francisco": {"temp_c": 18, "condition": "Foggy", "humidity": 78},
        "New York":      {"temp_c": 25, "condition": "Sunny", "humidity": 55},
        "London":        {"temp_c": 14, "condition": "Rainy", "humidity": 85},
        "Tokyo":         {"temp_c": 28, "condition": "Partly Cloudy", "humidity": 65},
        "Berlin":        {"temp_c": 12, "condition": "Overcast", "humidity": 72},
    }
    data = weather_data.get(city, {"temp_c": 20, "condition": "Clear", "humidity": 60})
    temp = data["temp_c"]
    if units == "fahrenheit":
        temp = round(temp * 9 / 5 + 32)
    return {"city": city, "temperature": temp, "units": units,
            "condition": data["condition"], "humidity": data["humidity"]}


def search_database(query: str, category: str = "all", max_results: int = 3) -> dict:
    """Simulate a product database search."""
    products = [
        {"name": "Wireless Headphones Pro",  "price": 79.99, "category": "electronics", "rating": 4.5},
        {"name": "Bluetooth Speaker Mini",   "price": 34.99, "category": "electronics", "rating": 4.2},
        {"name": "USB-C Hub 7-in-1",         "price": 45.00, "category": "electronics", "rating": 4.7},
        {"name": "Python Cookbook",           "price": 39.99, "category": "books",       "rating": 4.8},
        {"name": "ML Engineering",           "price": 54.99, "category": "books",       "rating": 4.6},
        {"name": "Running Jacket",           "price": 89.00, "category": "clothing",    "rating": 4.3},
    ]
    q = query.lower()
    results = [p for p in products if q in p["name"].lower() or q in p["category"]]
    if category != "all":
        results = [p for p in results if p["category"] == category]
    return {"query": query, "results": results[:max_results], "total_found": len(results)}


def calculate(expression: str) -> dict:
    """Safely evaluate a mathematical expression."""
    allowed = {"abs": abs, "round": round, "min": min, "max": max,
               "pow": pow, "sqrt": math.sqrt, "pi": math.pi, "e": math.e}
    try:
        result = eval(expression, {"__builtins__": {}}, allowed)
        return {"expression": expression, "result": result}
    except Exception as e:
        return {"expression": expression, "error": str(e)}


# Registry maps function names to implementations
TOOL_REGISTRY: dict[str, Callable] = {
    "get_weather": get_weather,
    "search_database": search_database,
    "calculate": calculate,
}

# Quick tests
print("Weather:  ", json.dumps(get_weather("San Francisco"), indent=2))
print("Search:   ", json.dumps(search_database("electronics"), indent=2))
print("Calculate:", json.dumps(calculate("2 ** 10 + 24"), indent=2))

---
## 5. Single Tool Call

The simplest pattern: send a user message, the model decides to call one tool,
we execute it and return the result so the model can produce a final answer.

In [ ]:
def single_tool_call(user_message: str):
    """Demonstrate a single tool-call round trip."""
    messages = [{"role": "user", "content": user_message}]

    # Step 1 -- ask the model (it may decide to call a tool)
    try:
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools, tool_choice="auto"
        )
        assistant_msg = response.choices[0].message
    except Exception as e:
        print(f"API call failed: {e}")
        print("Using mock response for demonstration.")
        # Mock: simulate the model requesting get_weather
        assistant_msg = type("M", (), {
            "role": "assistant", "content": None,
            "tool_calls": [type("TC", (), {
                "id": "call_mock_001", "type": "function",
                "function": type("F", (), {
                    "name": "get_weather",
                    "arguments": json.dumps({"city": "San Francisco", "units": "celsius"})
                })()
            })()]
        })()

    if not assistant_msg.tool_calls:
        print(f"Model answered directly: {assistant_msg.content}")
        return

    # Step 2 -- execute the tool
    tc = assistant_msg.tool_calls[0]
    fn_name = tc.function.name
    fn_args = json.loads(tc.function.arguments)
    print(f"Model requested tool: {fn_name}({fn_args})")

    result = TOOL_REGISTRY[fn_name](**fn_args)
    print(f"Tool result: {json.dumps(result, indent=2)}")

    # Step 3 -- send the result back
    messages.append({"role": "assistant", "content": None,
                     "tool_calls": [{"id": tc.id, "type": "function",
                                     "function": {"name": fn_name,
                                                  "arguments": tc.function.arguments}}]})
    messages.append({"role": "tool", "tool_call_id": tc.id,
                     "content": json.dumps(result)})

    try:
        final = client.chat.completions.create(model=MODEL, messages=messages)
        answer = final.choices[0].message.content
    except Exception as e:
        print(f"API call failed: {e}")
        answer = "Mock: San Francisco is currently 18 C and Foggy with 78 % humidity. Bring a jacket!"
        print("Using mock response for demonstration.")

    print(f"\nFinal answer: {answer}")


single_tool_call("What is the weather like in San Francisco?")

---
## 6. Multi-Tool Orchestration

Some queries require calling multiple tools. The model may request several
tool calls in a single response (parallel tool calls).

In [ ]:
def multi_tool_call(user_message: str):
    """Handle multiple parallel tool calls from one model response."""
    messages = [{"role": "user", "content": user_message}]

    try:
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools, tool_choice="auto"
        )
        assistant_msg = response.choices[0].message
    except Exception as e:
        print(f"API call failed: {e}")
        print("Using mock response for demonstration.")
        # Mock: model requests two weather calls in parallel
        def _mock_tc(id_, name, args):
            return type("TC", (), {
                "id": id_, "type": "function",
                "function": type("F", (), {"name": name, "arguments": json.dumps(args)})()
            })()
        assistant_msg = type("M", (), {
            "role": "assistant", "content": None,
            "tool_calls": [
                _mock_tc("call_m1", "get_weather", {"city": "New York", "units": "fahrenheit"}),
                _mock_tc("call_m2", "get_weather", {"city": "London", "units": "celsius"}),
            ]
        })()

    if not assistant_msg.tool_calls:
        print(f"Direct answer: {assistant_msg.content}")
        return

    print(f"Model requested {len(assistant_msg.tool_calls)} tool calls:\n")

    # Build assistant message for conversation history
    tc_data = [{"id": tc.id, "type": "function",
                "function": {"name": tc.function.name,
                             "arguments": tc.function.arguments}}
               for tc in assistant_msg.tool_calls]
    messages.append({"role": "assistant", "content": None, "tool_calls": tc_data})

    # Execute each tool call
    for tc in assistant_msg.tool_calls:
        fn_name = tc.function.name
        fn_args = json.loads(tc.function.arguments)
        print(f"  Calling {fn_name}({fn_args})")
        result = TOOL_REGISTRY[fn_name](**fn_args)
        print(f"  Result: {json.dumps(result)}\n")
        messages.append({"role": "tool", "tool_call_id": tc.id,
                         "content": json.dumps(result)})

    # Get final synthesis
    try:
        final = client.chat.completions.create(model=MODEL, messages=messages)
        answer = final.choices[0].message.content
    except Exception as e:
        print(f"API call failed: {e}")
        answer = ("Mock: New York is 77 F and Sunny while London is 14 C and Rainy. "
                  "New York is significantly warmer today!")
        print("Using mock response for demonstration.")

    print(f"Final answer: {answer}")


multi_tool_call("Compare the weather in New York and London right now.")

---
## 7. Tool Result Handling

Standardized result wrappers ensure every tool call returns a consistent
structure, making error handling uniform across the system.

In [ ]:
@dataclass
class ToolResult:
    """Standardized wrapper for tool execution results."""
    tool_call_id: str
    name: str
    success: bool
    data: Any = None
    error: str = ""

    def to_message(self) -> dict:
        """Convert to OpenAI tool-message format."""
        if self.success:
            content = json.dumps(self.data)
        else:
            content = json.dumps({"error": self.error,
                                  "suggestion": "Try a different approach."})
        return {"role": "tool", "tool_call_id": self.tool_call_id,
                "content": content}

    def __repr__(self):
        return f"ToolResult({self.name}, {'OK' if self.success else 'FAIL'})"


def execute_tool_safely(tool_call) -> ToolResult:
    """Execute a tool call with comprehensive error handling."""
    fn_name = tool_call.function.name

    # 1. Parse arguments
    try:
        fn_args = json.loads(tool_call.function.arguments)
    except json.JSONDecodeError as e:
        return ToolResult(tool_call.id, fn_name, False,
                          error=f"Invalid JSON arguments: {e}")

    # 2. Check registry
    if fn_name not in TOOL_REGISTRY:
        return ToolResult(tool_call.id, fn_name, False,
                          error=f"Unknown tool: {fn_name}")

    # 3. Execute
    try:
        result = TOOL_REGISTRY[fn_name](**fn_args)
        return ToolResult(tool_call.id, fn_name, True, data=result)
    except TypeError as e:
        return ToolResult(tool_call.id, fn_name, False,
                          error=f"Invalid arguments: {e}")
    except Exception as e:
        return ToolResult(tool_call.id, fn_name, False,
                          error=f"Execution error: {e}")


# Demonstrate with valid and invalid tool calls
def _fake_tc(id_, name, args_str):
    return type("TC", (), {
        "id": id_,
        "function": type("F", (), {"name": name, "arguments": args_str})()
    })()

test_cases = [
    _fake_tc("t1", "calculate",    json.dumps({"expression": "2**10"})),
    _fake_tc("t2", "unknown_tool", json.dumps({})),
    _fake_tc("t3", "get_weather",  "{bad json"),
    _fake_tc("t4", "get_weather",  json.dumps({"wrong_param": 123})),
]

for tc in test_cases:
    r = execute_tool_safely(tc)
    print(f"{r}  ->  {r.to_message()['content'][:80]}")

---
## 8. Agent Loop (Observe-Think-Act)

A full agent loop runs iteratively. The model *observes* (reads messages),
*thinks* (decides next action), and *acts* (calls a tool or responds).
The loop continues until the model produces a final text response or
we hit `max_iterations`.

In [ ]:
def agent_loop(user_message: str, max_iterations: int = 5,
               verbose: bool = True) -> str:
    """Run an observe-think-act agent loop."""
    messages = [
        {"role": "system", "content": (
            "You are a helpful assistant with access to tools. "
            "Use tools when needed to answer accurately. "
            "You may call multiple tools if the task requires it."
        )},
        {"role": "user", "content": user_message}
    ]

    for iteration in range(max_iterations):
        if verbose:
            print(f"\n--- Iteration {iteration + 1} ---")

        # OBSERVE + THINK
        try:
            response = client.chat.completions.create(
                model=MODEL, messages=messages, tools=tools, tool_choice="auto"
            )
            msg = response.choices[0].message
        except Exception as e:
            print(f"API call failed: {e}")
            print("Using mock response for demonstration.")
            if iteration == 0:
                msg = type("M", (), {
                    "role": "assistant", "content": None,
                    "tool_calls": [_fake_tc(f"c{iteration}", "search_database",
                                            json.dumps({"query": "electronics", "category": "electronics"}))]
                })()
            elif iteration == 1:
                msg = type("M", (), {
                    "role": "assistant", "content": None,
                    "tool_calls": [_fake_tc(f"c{iteration}", "calculate",
                                            json.dumps({"expression": "79.99 + 34.99 + 45.00"}))]
                })()
            else:
                msg = type("M", (), {
                    "role": "assistant",
                    "content": ("Mock: I found 3 electronics products totaling $159.98: "
                                "Wireless Headphones Pro ($79.99), Bluetooth Speaker Mini "
                                "($34.99), and USB-C Hub 7-in-1 ($45.00)."),
                    "tool_calls": None
                })()

        # ACT -- if no tool calls we have the final answer
        if not msg.tool_calls:
            if verbose:
                print(f"Agent finished after {iteration + 1} iteration(s).")
            print(f"\nFinal answer: {msg.content}")
            return msg.content

        # ACT -- execute tool calls
        tc_data = [{"id": tc.id, "type": "function",
                    "function": {"name": tc.function.name,
                                 "arguments": tc.function.arguments}}
                   for tc in msg.tool_calls]
        messages.append({"role": "assistant", "content": None,
                         "tool_calls": tc_data})

        for tc in msg.tool_calls:
            result = execute_tool_safely(tc)
            if verbose:
                print(f"  Tool: {result}")
            messages.append(result.to_message())

    print("Agent reached max iterations without completing.")
    return ""


agent_loop("Find all electronics products and calculate their total price.")

---
## 9. Error Handling in Tool Execution

Production agents need retries, timeouts, structured logging, and
execution statistics.

In [ ]:
class RobustToolExecutor:
    """Tool executor with retries, logging, and metrics."""

    def __init__(self, registry: dict, max_retries: int = 2):
        self.registry = registry
        self.max_retries = max_retries
        self.log: list[dict] = []

    def execute(self, tool_call) -> ToolResult:
        fn_name = tool_call.function.name
        t0 = time.time()

        if fn_name not in self.registry:
            self._record(fn_name, False, 0, "Unknown tool")
            return ToolResult(tool_call.id, fn_name, False,
                              error=f"Unknown tool: {fn_name}")

        try:
            fn_args = json.loads(tool_call.function.arguments)
        except json.JSONDecodeError as e:
            self._record(fn_name, False, 0, f"Bad JSON: {e}")
            return ToolResult(tool_call.id, fn_name, False,
                              error=f"Invalid arguments: {e}")

        last_err = None
        for attempt in range(self.max_retries + 1):
            try:
                data = self.registry[fn_name](**fn_args)
                elapsed = time.time() - t0
                self._record(fn_name, True, elapsed)
                return ToolResult(tool_call.id, fn_name, True, data=data)
            except Exception as e:
                last_err = e
                if attempt < self.max_retries:
                    print(f"  Retry {attempt+1} for {fn_name}: {e}")

        elapsed = time.time() - t0
        self._record(fn_name, False, elapsed, str(last_err))
        return ToolResult(tool_call.id, fn_name, False, error=str(last_err))

    def _record(self, name, ok, elapsed, error=None):
        self.log.append({"tool": name, "success": ok,
                         "elapsed_ms": round(elapsed * 1000, 1),
                         "error": error})

    def stats(self) -> dict:
        n = len(self.log)
        ok = sum(1 for e in self.log if e["success"])
        avg = sum(e["elapsed_ms"] for e in self.log) / max(n, 1)
        return {"total": n, "success": ok, "fail": n - ok,
                "avg_ms": round(avg, 1)}


executor = RobustToolExecutor(TOOL_REGISTRY)

demo_calls = [
    _fake_tc("r1", "calculate",    json.dumps({"expression": "100 / 3"})),
    _fake_tc("r2", "get_weather",  json.dumps({"city": "Tokyo"})),
    _fake_tc("r3", "nonexistent",  json.dumps({})),
]
for tc in demo_calls:
    print(executor.execute(tc))

print(f"\nStats: {json.dumps(executor.stats(), indent=2)}")

---
## 10. Complete Example: Travel Planning Agent

A travel-planning agent that combines weather checks, product searches,
and cost calculations in a single conversation.

In [ ]:
class TravelAgent:
    """End-to-end travel planning agent using function calling."""

    def __init__(self):
        self.executor = RobustToolExecutor(TOOL_REGISTRY)
        self.conversation = [
            {"role": "system", "content": (
                "You are a travel planning assistant. Use tools to:\n"
                "1. Check weather at destinations\n"
                "2. Search for travel products\n"
                "3. Calculate costs and budgets\n"
                "Always provide specific, data-backed recommendations."
            )}
        ]

    def chat(self, user_message: str) -> str:
        self.conversation.append({"role": "user", "content": user_message})

        for _ in range(5):
            try:
                resp = client.chat.completions.create(
                    model=MODEL, messages=self.conversation,
                    tools=tools, tool_choice="auto"
                )
                msg = resp.choices[0].message
            except Exception as e:
                print(f"API call failed: {e}")
                print("Using mock response for demonstration.")
                answer = (
                    "Mock: Based on my research:\n"
                    "- Tokyo: 28 C, Partly Cloudy -- great for sightseeing\n"
                    "- San Francisco: 18 C, Foggy -- pack layers\n\n"
                    "Recommended travel electronics:\n"
                    "- Wireless Headphones Pro: $79.99\n"
                    "- USB-C Hub 7-in-1: $45.00\n"
                    "Total gear budget: $124.99"
                )
                self.conversation.append({"role": "assistant", "content": answer})
                return answer

            if not msg.tool_calls:
                self.conversation.append({"role": "assistant",
                                          "content": msg.content})
                return msg.content

            tc_data = [{"id": tc.id, "type": "function",
                        "function": {"name": tc.function.name,
                                     "arguments": tc.function.arguments}}
                       for tc in msg.tool_calls]
            self.conversation.append({"role": "assistant", "content": None,
                                      "tool_calls": tc_data})
            for tc in msg.tool_calls:
                r = self.executor.execute(tc)
                self.conversation.append(r.to_message())

        return "Could not complete the request within the iteration limit."


agent = TravelAgent()
print(agent.chat(
    "I am planning a trip. Check the weather in Tokyo and San Francisco, "
    "and find travel electronics I might need."
))
print(f"\nTool stats: {json.dumps(agent.executor.stats(), indent=2)}")

---
## Summary

This notebook covered the complete function calling / tool use pattern for LLM agents:

1. **Tool Schemas** -- Define tools as JSON schemas with clear descriptions and typed parameters.
2. **Single Tool Call** -- The basic request-execute-respond flow.
3. **Multi-Tool Orchestration** -- Handling parallel tool calls in one model response.
4. **Result Handling** -- Standardized `ToolResult` wrappers for consistent error reporting.
5. **Agent Loop** -- The observe-think-act pattern for multi-step reasoning with iteration limits.
6. **Error Handling** -- Retries, logging, and execution statistics for production readiness.

### Key Takeaways

- Always validate tool arguments before execution.
- Use a registry pattern to map function names to implementations.
- Wrap results in a standard format so the model can handle errors gracefully.
- Set iteration limits on agent loops to prevent runaway execution.
- Log every tool call for observability and debugging.